In [1]:
import sys
import os
import pandas as pd
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
import clean.cleaning as cd

train_data = pd.read_parquet("./data/block1_train.parquet")
test_data  = pd.read_parquet("./data/block2_test.parquet")

#train_data = cd.clean_all(train_data)

In [ ]:
import sys
import os
import pandas as pd
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
import clean.cleaning as cd
import feats.features as ft
import numpy as np

train_data = pd.read_parquet("data/block1_train.parquet")
test_data  = pd.read_parquet("data/block2_test.parquet")

#train_data = cd.clean_all(train_data)
train_data = ft.features_all(train_data)
test_data = ft.features_all(test_data)

✅ Kreiran: urban_amenities_score
✅ Kreiran: vehicle_score
UKLANJANJE VISOKO KORELISANIH FEATURE-A (prag: 0.97)
Broj numeričkih kolona za proveru: 12

✅ Nema kolona za izbacivanje (sve korelacije ispod praga)
✅ Kreiran: urban_amenities_score
UKLANJANJE VISOKO KORELISANIH FEATURE-A (prag: 0.97)
Broj numeričkih kolona za proveru: 12

✅ Nema kolona za izbacivanje (sve korelacije ispod praga)


In [3]:
train_data['urban_amenities_score']

0         0.160405
1         0.645913
2         0.232072
3         0.645913
4         0.208181
            ...   
541287    0.058340
541288    0.350490
541289    0.091038
541290    0.101912
541291    0.058340
Name: urban_amenities_score, Length: 397623, dtype: float64

In [4]:
target_cols = [f"Insurer_{chr(i)}_price" for i in range(ord('A'), ord('K') + 1)]
deductible_cols = [f"Insurer_{chr(i)}_deductible" for i in range(ord('A'), ord('K') + 1)]

In [5]:
import pandas as pd
import numpy as np

def detect_categorical_columns(df):
    categorical_cols = []
    
    for col in df.columns:
        # Uzimamo samo vrednosti koje nisu NaN za testiranje
        non_null_values = df[col].dropna()
        
        if non_null_values.empty:
            # Ako je cela kolona prazna, tretiramo je kao kategorijsku ili je brišemo
            categorical_cols.append(col)
            continue
            
        try:
            # Pokušavamo da konvertujemo celu kolonu (bez NaN) u float
            pd.to_numeric(non_null_values, errors='raise')
        except (ValueError, TypeError):
            # Ako baci grešku, znači da ima teksta koji nije broj
            categorical_cols.append(col)
            
    return categorical_cols

# Korišćenje:
cat_cols = detect_categorical_columns(train_data)
print(cat_cols)

['vehicle_number_plate', 'coverage', 'payment_frequency', 'contractor_birthdate', 'vehicle_maker', 'vehicle_model', 'vehicle_fuel_type', 'vehicle_primary_color', 'vehicle_first_registration_date', 'vehicle_country_first_registration_date', 'vehicle_last_registration_date', 'vehicle_inspection_report_date', 'vehicle_inspection_expiry_date', 'vehicle_odometer_verdict_code', 'vehicle_is_imported', 'vehicle_is_imported_within_last_12_months', 'vehicle_has_open_recall', 'province', 'municipality', 'driver_age_band']


In [6]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

def encode_categorical(train, test, cat_cols, high_sparsity_threshold=10):

    train_enc = train.copy()
    test_enc = test.copy()
    
    high_sparse_cols = [c for c in cat_cols if train[c].nunique() > high_sparsity_threshold]
    low_sparse_cols = [c for c in cat_cols if train[c].nunique() <= high_sparsity_threshold]
    
    # Ordinal encoding za high sparsity
    if high_sparse_cols:
        oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        train_enc[high_sparse_cols] = oe.fit_transform(train[high_sparse_cols])
        test_enc[high_sparse_cols] = oe.transform(test[high_sparse_cols])
    
    # One-hot encoding za low sparsity
    if low_sparse_cols:
        ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        train_ohe = ohe.fit_transform(train[low_sparse_cols])
        test_ohe = ohe.transform(test[low_sparse_cols])
        
        # Napravimo kolone sa imenima
        ohe_cols = ohe.get_feature_names_out(low_sparse_cols)
        train_ohe_df = pd.DataFrame(train_ohe, columns=ohe_cols, index=train.index)
        test_ohe_df = pd.DataFrame(test_ohe, columns=ohe_cols, index=test.index)
        
        # Drop original low sparsity cols i dodaj one-hot
        train_enc = train_enc.drop(columns=low_sparse_cols).join(train_ohe_df)
        test_enc = test_enc.drop(columns=low_sparse_cols).join(test_ohe_df)
    
    return train_enc, test_enc

In [7]:
train_data_enc, test_data_enc = encode_categorical(train_data, test_data, cat_cols)

In [8]:
object_cols = train_data_enc.select_dtypes(include=['object']).columns

C:\Users\velja\AppData\Local\Temp\ipykernel_36656\4049719601.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = train_data_enc.select_dtypes(include=['object']).columns


In [9]:
# 1. Uzimamo sve kolone koje su trenutno tipa 'object'
object_cols = train_data_enc.select_dtypes(include=['object']).columns

print(f"Započinjem konverziju {len(object_cols)} kolona...")

for col in object_cols:
    # Konvertujemo u numerik (float64 po defaultu)
    # errors='coerce' sprečava pucanje koda ako naiđe na tekstualni bag
    train_data_enc[col] = pd.to_numeric(train_data_enc[col], errors='coerce')

    # Isto radimo i za test set da bi struktura ostala identična
    if col in test_data_enc.columns:
        test_data_enc[col] = pd.to_numeric(test_data_enc[col], errors='coerce')

# 2. Finalna provera tipova
print("-" * 30)
print("Konverzija gotova!")
print(f"Preostalo 'object' kolona u train: {len(train_data_enc.select_dtypes(include=['object']).columns)}")

C:\Users\velja\AppData\Local\Temp\ipykernel_36656\1673138626.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = train_data_enc.select_dtypes(include=['object']).columns


Započinjem konverziju 102 kolona...
------------------------------
Konverzija gotova!
Preostalo 'object' kolona u train: 0


In [10]:
# from catboost import CatBoostRegressor
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import mean_absolute_error, r2_score
# import pandas as pd
# import numpy as np

# # 1. Podela podataka
# X = train_data_enc.drop(columns=target_cols, errors='ignore').copy()
# Y = train_data[target_cols]

# X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=42)

# trained_models = {}

# print("Počinjem trening po targetima sa CatBoost-om...")

# for col in target_cols:
#     print(f"--- Treniram CatBoost model za: {col} ---")
    
#     # Filtriranje validnih redova (tamo gde target nije NaN)
#     valid_mask = Y_train[col].notna()
#     X_train_sub = X_train[valid_mask]
#     y_train_sub = Y_train.loc[valid_mask, col]
    
#     # 2. CatBoost Model
#     # 'allow_writing_files=False' sprečava CatBoost da pravi foldere sa logovima u svakom krugu petlje
#     model = CatBoostRegressor(
#         iterations=1000,
#         learning_rate=0.05,
#         depth=6,
#         loss_function='MAE', # Možeš ostaviti 'RMSE', ali MAE je često stabilniji za osiguranje
#         random_seed=42,
#         verbose=100,         # Ispisuje napredak na svakih 100 stabala
#         allow_writing_files=False,
#         thread_count=-1      # Koristi sve CPU jezgre (ekvivalent za n_jobs=-1)
#     )
    
#     # Treniranje
#     model.fit(X_train_sub, y_train_sub)
#     trained_models[col] = model
    
#     # 3. Dodavanje predviđanja kao feature-a za naredne iteracije
#     # .predict() u CatBoost-u radi identično kao u XGBoost-u
#     X_train[f"pred_{col}"] = model.predict(X_train)
#     X_val[f"pred_{col}"] = model.predict(X_val)
    
#     print(f"Model za {col} je završen i dodat kao feature.")

# print("\nSvi CatBoost modeli su istrenirani!")

In [11]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import pandas as pd
import numpy as np

target_cols = [f"Insurer_{chr(i)}_price" for i in range(ord('A'), ord('K') + 1)]

# Čistimo X setove od originalnih cena (da model ne bi "varao")
X_train_current = train_data_enc.drop(columns=target_cols, errors='ignore').copy()
X_test_current = test_data_enc.drop(columns=target_cols, errors='ignore').copy()
Y_train = train_data_enc[target_cols]

# Sređivanje kategoričkih kolona za CatBoost (mora biti string)
# Pretpostavljamo da je cat_cols lista tvojih kategoričkih feature-a
current_cat_features = list(set(cat_cols).intersection(X_train_current.columns))
for c in current_cat_features:
    X_train_current[c] = X_train_current[c].fillna("Missing").astype(str)
    X_test_current[c] = X_test_current[c].fillna("Missing").astype(str)

test_predictions = {}
trained_models = {}

print("🚀 Počinjem CatBoost trening sa Log-Transformacijom i Chaining-om...")

for col in target_cols:
    if col not in Y_train.columns:
        continue
        
    print(f"\n🎯 Obrađujem: {col}")
    
    # Uzimamo samo redove gde imamo cenu za taj Insurer
    valid_mask = Y_train[col].notna()
    X_sub = X_train_current.loc[valid_mask].copy()
    
    # LOG TRANSFORMACIJA targeta: log(1 + y)
    y_sub_log = np.log1p(Y_train.loc[valid_mask, col])
    
    # 1. BRZA SELEKCIJA FEATURE-A (da bismo ubrzali chaining)
    selector = CatBoostRegressor(
        iterations=200, 
        learning_rate=0.1, 
        depth=6, 
        verbose=0, 
        allow_writing_files=False, 
        random_seed=42
    )
    selector.fit(X_sub, y_sub_log, cat_features=current_cat_features)
    
    # Uzimamo top feature-e koji čine 95% važnosti
    fi = selector.get_feature_importance()
    fn = np.array(selector.feature_names_)
    sorted_idx = np.argsort(fi)[::-1]
    cum_importance = np.cumsum(fi[sorted_idx])
    final_num = max(5, min(np.where(cum_importance >= 95.0)[0][0] + 1, 50))
    dynamic_features = fn[sorted_idx][:final_num].tolist()
    
    # 2. FINALNI TRENING NA LOG SKALI
    final_cat = [c for c in current_cat_features if c in dynamic_features]
    
    model = CatBoostRegressor(
        iterations=1500, 
        learning_rate=0.05, 
        depth=6, 
        loss_function='RMSE', 
        verbose=200, 
        allow_writing_files=False,
        random_seed=42
    )
    
    model.fit(X_sub[dynamic_features], y_sub_log, cat_features=final_cat)
    trained_models[col] = model
    
    # 3. PREDVIĐANJE I REVERZIJA (Back to Normal)
    # Predviđamo za ceo Train i Test set
    train_log_preds = model.predict(X_train_current[dynamic_features])
    test_log_preds = model.predict(X_test_current[dynamic_features])
    
    # Vraćamo iz logaritma u normalne cene: exp(x) - 1
    train_original_preds = np.expm1(train_log_preds)
    test_original_preds = np.expm1(test_log_preds)
    
    # 4. DODAVANJE PREDVIĐANJA KAO FEATURE ZA SLEDEĆI MODEL (Chaining)
    X_train_current[f"pred_{col}"] = train_original_preds
    X_test_current[f"pred_{col}"] = test_original_preds
    
    # Čuvamo za finalni fajl
    test_predictions[col] = np.round(test_original_preds, 2)
    
    print(f"✅ Završen {col}")

# 5. KREIRANJE SUBMISSION FAJLA
submission = pd.DataFrame(test_predictions)

# Dodajemo quote_id sa originalnog test_data (proveri da li se zove baš test_data)
submission.insert(0, 'quote_id', test_data['quote_id'].astype(int).values)

# Snimanje u CSV (identičan format kao tvoj primer)
submission.to_csv('catboost_log_submission.csv', index=False, sep=';')

print("\n🏁 PROCES ZAVRŠEN!")
print(submission.head())

🚀 Počinjem CatBoost trening sa Log-Transformacijom i Chaining-om...

🎯 Obrađujem: Insurer_A_price
0:	learn: 0.6515558	total: 197ms	remaining: 4m 56s
200:	learn: 0.1506229	total: 30.4s	remaining: 3m 16s
400:	learn: 0.1381737	total: 59.4s	remaining: 2m 42s
600:	learn: 0.1327263	total: 1m 28s	remaining: 2m 13s
800:	learn: 0.1292125	total: 1m 59s	remaining: 1m 44s
1000:	learn: 0.1266093	total: 2m 30s	remaining: 1m 14s
1200:	learn: 0.1245944	total: 3m 1s	remaining: 45.1s
1400:	learn: 0.1228060	total: 3m 31s	remaining: 14.9s
1499:	learn: 0.1220819	total: 3m 45s	remaining: 0us
✅ Završen Insurer_A_price

🎯 Obrađujem: Insurer_B_price
0:	learn: 0.6036133	total: 142ms	remaining: 3m 32s
200:	learn: 0.1347998	total: 25.3s	remaining: 2m 43s
400:	learn: 0.1246388	total: 50.3s	remaining: 2m 17s
600:	learn: 0.1198490	total: 1m 15s	remaining: 1m 52s
800:	learn: 0.1169135	total: 1m 40s	remaining: 1m 27s
1000:	learn: 0.1145683	total: 2m 5s	remaining: 1m 2s
1200:	learn: 0.1127672	total: 2m 30s	remaining: 3

In [12]:
importance_data = []

for target_name, model in trained_models.items():
    # Izvlačenje važnosti i imena feature-a direktno iz modela
    fi = model.get_feature_importance()
    fn = model.feature_names_
    
    # Kreiranje privremenog DataFrame-a za ovaj model
    temp_df = pd.DataFrame({
        'Target': target_name,
        'Feature': fn,
        'Importance': fi
    })
    
    # Sortiramo da najbitniji budu na vrhu za taj target
    temp_df = temp_df.sort_values(by='Importance', ascending=False)
    
    importance_data.append(temp_df)

# Spajamo sve u jedan veliki DataFrame
df_importance_final = pd.concat(importance_data, ignore_index=True)

# Eksport u CSV
df_importance_final.to_csv('model_feature_importances.csv', index=False)

print("✅ CSV sa važnošću feature-a je sačuvan pod imenom 'model_feature_importances.csv'")

# Prikaz top 5 za svaki target u konzoli radi provere
print(df_importance_final.groupby('Target').head(5))

✅ CSV sa važnošću feature-a je sačuvan pod imenom 'model_feature_importances.csv'
              Target                       Feature  Importance
0    Insurer_A_price              claim_free_years   14.780225
1    Insurer_A_price         driver_age_band_18-24   11.998961
2    Insurer_A_price      payment_frequency_yearly    9.012149
3    Insurer_A_price     payment_frequency_monthly    8.706104
4    Insurer_A_price  cfy_standardized_by_province    5.790712
30   Insurer_B_price          pred_Insurer_A_price   66.956475
31   Insurer_B_price                coverage_casco    4.392449
32   Insurer_B_price  cfy_standardized_by_province    4.043845
33   Insurer_B_price              claim_free_years    3.388581
34   Insurer_B_price                   vehicle_age    2.949522
59   Insurer_C_price          pred_Insurer_A_price   50.624258
60   Insurer_C_price          pred_Insurer_B_price   19.780184
61   Insurer_C_price                 vehicle_maker    3.371805
62   Insurer_C_price                

In [13]:
# import matplotlib.pyplot as plt
# target_to_plot = target_cols[-1] # Primer za poslednji target
# current_model = trained_models[target_to_plot]

# # CatBoost čuva imena feature-a na kojima je treniran!
# # Ovo je NAJSIGURNIJI način da izbegneš IndexError
# feature_names = current_model.feature_names_
# importances = current_model.get_feature_importance()

# # Sortiranje
# sorted_indices = np.argsort(importances)

# # Plotovanje
# plt.figure(figsize=(10, 20)) # Povećaj visinu jer imaš 100+ kolona
# plt.barh(np.array(feature_names)[sorted_indices], importances[sorted_indices])
# plt.xlabel('CatBoost Feature Importance')
# plt.title(f'Važnost kolona za target: {target_to_plot}')
# plt.show() 

In [14]:
feature_names = np.array(final_model.feature_names_)
importances = final_model.get_feature_importance()

# Sortiramo indekse od najmanjeg ka najvećem i uzimamo poslednjih 20
top_20_indices = np.argsort(importances)[-20:]
top_20_features = feature_names[top_20_indices].tolist()

print("Top 20 kolona za novi trening:")
print(top_20_features)

X_train_top = X_train[top_20_features]
X_val_top = X_val[top_20_features]

NameError: name 'final_model' is not defined

In [ ]:
model_top = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function='MAE',
    random_seed=42,
    verbose=100,
    allow_writing_files=False
)

# Ako i dalje imaš kategoričke kolone među tih top 20, 
# moraš ih ponovo navesti (samo one koje su ostale u top 20)
new_cat_features = [col for col in cat_cols if col in top_20_features]

model_top.fit(
    X_train_top, Y_train[col], # Koristiš target iz petlje ili specifičan
    eval_set=(X_val_top, Y_val[col]),
    cat_features=new_cat_features
)

CatBoostError: catboost/libs/metrics/metric.cpp:7032: metric/loss-function MAE does not allow nan values in target data

In [ ]:
from sklearn.metrics import r2_score
results = []

print("\n--- EVALUACIJA MODELA PO OSIGURAVAČIMA ---")

for col in target_cols:
    # 1. Provera da li predviđanje uopšte postoji u X_val
    pred_col_name = f"pred_{col}"
    if pred_col_name not in X_val.columns:
        print(f"Preskačem {col} - predviđanje nije pronađeno.")
        continue
    
    # 2. Uzimanje predviđenih i stvarnih vrednosti
    y_pred = X_val[pred_col_name]
    y_true = Y_val[col]
    
    # 3. FILTRIRANJE: Ovo je ključno jer CatBoost nije treniran na NaN vrednostima targeta
    # Moramo evaluirati samo tamo gde smo imali šta da uporedimo (gde stvarna cena postoji)
    mask = y_true.notna()
    
    if mask.sum() > 0:
        actuals = y_true[mask]
        predictions = y_pred[mask]
        
        mae = mean_absolute_error(actuals, predictions)
        r2 = r2_score(actuals, predictions)
        
        # Opciono: Dodajemo i Mean Price da vidimo koliki je MAE u odnosu na prosečnu cenu
        mean_val = actuals.mean()
        error_pct = (mae / mean_val) * 100 if mean_val != 0 else 0
        
        results.append({
            'Osiguravač': col.replace('Insurer_', '').replace('_price', ''),
            'MAE': round(mae, 2),
            'R2': round(r2, 4),
            'Error %': round(error_pct, 2)
        })

# 4. Prikaz rezultata u tabeli
df_results = pd.DataFrame(results)

# Sortiramo po MAE (od najboljeg ka najgorem modelu)
df_results = df_results.sort_values(by='MAE')

print(df_results.to_string(index=False))

print("-" * 40)
print(f"Prosečan MAE svih osiguravača: {df_results['MAE'].mean():.2f}")
print(f"Prosečan R2 score: {df_results['R2'].mean():.4f}")


--- EVALUACIJA MODELA PO OSIGURAVAČIMA ---
Osiguravač   MAE     R2  Error %
         G  6.90 0.9535     8.15
         C  8.54 0.9418     9.57
         J  9.98 0.9394     9.53
         B 10.18 0.9372    10.18
         D 10.31 0.9383    10.44
         A 10.56 0.9548    10.85
         F 10.96 0.9131    11.21
         H 12.46 0.9060    10.74
         K 12.93 0.9319    10.39
         I 14.52 0.8635    15.32
         E 17.68 0.8394    15.24
----------------------------------------
Prosečan MAE svih osiguravača: 11.37
Prosečan R2 score: 0.9199


In [ ]:
print(f"\n📍 Najbitnijih 10 faktora za {col}:")
# Uzimamo top 10 radi preglednosti u konzoli, a sortiramo ih od najbitnijeg
# get_feature_importance() daje skorove, pa ih uparujemo sa imenima
top_scores = fi[np.argsort(fi)[-10:]][::-1]
top_names = fn[np.argsort(fi)[-10:]][::-1]

for name, score in zip(top_names, top_scores):
    print(f"  - {name}: {score:.2f}")


📍 Najbitnijih 10 faktora za Insurer_K_price:
  - pred_Insurer_D_price: 17.97
  - pred_Insurer_B_price: 16.82
  - pred_Insurer_E_price: 8.73
  - claim_free_years: 8.63
  - pred_Insurer_C_price: 5.68
  - Insurer_K_deductible: 4.28
  - vehicle_fuel_type_diesel: 4.21
  - pred_Insurer_F_price: 4.06
  - pred_Insurer_I_price: 4.02
  - pred_Insurer_J_price: 2.63


In [ ]:
import pandas as pd
from xgboost import XGBRegressor

# 1. PRIPREMA: Izbacujemo sve targete iz trening seta pre početka
# Moramo izbaciti 'Insurer_A_price', 'Insurer_B_price'... iz X setova
X_train_current = train_data_enc.drop(columns=target_cols, errors='ignore').copy()
X_test_current = test_data_enc.drop(columns=target_cols, errors='ignore').copy()

trained_models = {}
test_predictions = {}

print("Započeto treniranje...")

for col in target_cols:
    print(f"Obrađujem: {col}")
    
    # Maska za validne redove ostaje ista jer gledamo Y_train (originalne targete)
    valid_mask = Y_train[col].notna()
    
    # Uzimamo trenutne feature (oni sada NE sadrže originalne cene, samo predikcije)
    X_sub = X_train_current.loc[valid_mask]
    y_sub = Y_train.loc[valid_mask, col]
    
    model = XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        tree_method='hist',
        random_state=42,
        n_jobs=-1
    )
    
    # Fit modela na čistim podacima
    model.fit(X_sub, y_sub)
    trained_models[col] = model
    
    # DODAVANJE PREDVIĐANJA KAO FEATURE:
    # Ovo je jedini način na koji sledeći model "vidi" cene prethodnih
    train_preds = model.predict(X_train_current)
    X_train_current[f"pred_{col}"] = train_preds
    
    test_preds = model.predict(X_test_current)
    X_test_current[f"pred_{col}"] = test_preds
    
    test_predictions[col] = np.round(test_preds, 2)

submission = pd.DataFrame(test_predictions)

submission.insert(0, 'quote_id', test_data['quote_id'].astype(int).values)

submission.to_csv('finalni_rezultati.csv', index=False, sep=';')

print("\n--- PROCES ZAVRŠEN ---")
print("Prvih par redova tvog submission-a:")
print(submission.head())

Započeto treniranje...
Obrađujem: Insurer_A_price


IndexingError: Unalignable boolean Series provided as indexer (index of the boolean Series and of the indexed object do not match).

In [ ]:
test_data.shape

(164092, 144)

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import pandas as pd
import numpy as np

# 1. Priprema podataka
# Izbacujemo targete (cene) iz X seta pre podele, jer njih model ne sme da vidi kao ulaz
X = train_data_enc.drop(columns=target_cols, errors='ignore').copy()
Y = train_data[target_cols]

# Podela na Train i Val
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=42)

trained_models = {}

print("Počinjem trening po targetima (sa chaining metodom)...")

for col in target_cols:
    print(f"--- Treniram model za: {col} ---")
    
    # Koristimo samo redove gde trenutni target u Y_train nije NaN
    valid_mask = Y_train[col].notna()
    
    # Uzimamo trenutne feature (originalni + predviđanja prethodnih modela)
    X_train_sub = X_train.loc[valid_mask]
    y_train_sub = Y_train.loc[valid_mask, col]
    
    # 2. Model definicija
    model = XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        tree_method='hist',
        random_state=42,
        n_jobs=-1
    )
    
    # Trening
    model.fit(X_train_sub, y_train_sub)
    trained_models[col] = model
    
    # 3. DODAVANJE PREDVIĐANJA KAO NOVI FEATURE
    # Predviđamo za CEO X_train i X_val kako bi sledeći model imao ove podatke
    X_train[f"pred_{col}"] = model.predict(X_train)
    X_val[f"pred_{col}"] = model.predict(X_val)
    
    print(f"Model za {col} završen. Dodat feature 'pred_{col}' u Train i Val.")

print("\nSvi modeli su istrenirani! Sada možeš proveriti metrike.")

# --- OPCIONA PROVERA REZULTATA ---
results = []
for col in target_cols:
    y_true = Y_val[col]
    y_pred = X_val[f"pred_{col}"]
    mask = y_true.notna()
    
    mae = mean_absolute_error(y_true[mask], y_pred[mask])
    results.append({'Target': col, 'MAE': mae})

print("\nMAE po osiguravačima na validacionom setu:")
print(pd.DataFrame(results))

Počinjem trening po targetima (sa chaining metodom)...
--- Treniram model za: Insurer_A_price ---
Model za Insurer_A_price završen. Dodat feature 'pred_Insurer_A_price' u Train i Val.
--- Treniram model za: Insurer_B_price ---
Model za Insurer_B_price završen. Dodat feature 'pred_Insurer_B_price' u Train i Val.
--- Treniram model za: Insurer_C_price ---
Model za Insurer_C_price završen. Dodat feature 'pred_Insurer_C_price' u Train i Val.
--- Treniram model za: Insurer_D_price ---
Model za Insurer_D_price završen. Dodat feature 'pred_Insurer_D_price' u Train i Val.
--- Treniram model za: Insurer_E_price ---
Model za Insurer_E_price završen. Dodat feature 'pred_Insurer_E_price' u Train i Val.
--- Treniram model za: Insurer_F_price ---
Model za Insurer_F_price završen. Dodat feature 'pred_Insurer_F_price' u Train i Val.
--- Treniram model za: Insurer_G_price ---
Model za Insurer_G_price završen. Dodat feature 'pred_Insurer_G_price' u Train i Val.
--- Treniram model za: Insurer_H_price ---

In [ ]:
print(pd.DataFrame(results))

             Target        MAE
0   Insurer_A_price  12.445129
1   Insurer_B_price  11.612943
2   Insurer_C_price  10.796713
3   Insurer_D_price  12.269809
4   Insurer_E_price  18.605483
5   Insurer_F_price  11.956905
6   Insurer_G_price   7.447456
7   Insurer_H_price  15.039967
8   Insurer_I_price  16.337037
9   Insurer_J_price  13.882767
10  Insurer_K_price  13.787913
